# Assignment 4: K-Nearest Neighbors (KNN) Implementation
Student ID: 220119

This notebook implements **K-Nearest Neighbors (KNN)** using scikit-learn on the Teens Mental Health Dataset.
KNN is a distance-based, non-parametric, lazy learning algorithm used for classification.

## Dataset
- **Teens Mental Health Dataset**
- Target: depression_label (0 = Not Depressed, 1 = Depressed)
- Features: age, gender, daily_social_media_hours, platform_usage, sleep_hours, screen_time_before_sleep, academic_performance, physical_activity, social_interaction_level, stress_level, anxiety_level, addiction_level

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)
from sklearn.inspection import DecisionBoundaryDisplay

## 1. Load the Dataset

In [2]:
# Load Teens Mental Health Dataset
# Try local file first, then download from GitHub
try:
    df = pd.read_csv("Teen_Mental_Health_Dataset.csv")
except:
    url = "https://raw.githubusercontent.com/ahsanjust/Artificial-Intelligence-and-Machine-Learning-Lab/master/Lab_3/KNN/Teen_Mental_Health_Dataset.csv"
    df = pd.read_csv(url)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


Dataset loaded successfully!
Shape: (1200, 13)
Columns: ['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label']


### 1.1 Exploratory Data Analysis

In [3]:
# First few rows
print("First 5 rows of the dataset:")
display(df.head())

# Dataset info
print("\nDataset Info:")
df.info()

# Statistical summary
print("\nStatistical Summary:")
display(df.describe())

First 5 rows of the dataset:


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0



Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   str    
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   str    
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   str    
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   int64  
dtypes: float64(5), int64(5), str(3)
memory usage: 122.0 KB

Sta

,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,stress_level,anxiety_level,addiction_level,depression_label
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,15.928333,4.536667,6.449417,1.740333,2.990383,1.014500,5.445833,5.636667,5.565000,0.025833
std,2.021947,2.029599,1.442677,0.716660,0.576758,0.582185,2.903290,2.859453,2.830627,0.158704
min,13.000000,1.000000,4.000000,0.500000,2.000000,0.000000,1.000000,1.000000,1.000000,0.000000
25%,14.000000,2.800000,5.200000,1.100000,2.500000,0.500000,3.000000,3.000000,3.000000,0.000000
50%,16.000000,4.500000,6.500000,1.800000,2.990000,1.000000,5.000000,6.000000,6.000000,0.000000
75%,18.000000,6.300000,7.600000,2.400000,3.480000,1.500000,8.000000,8.000000,8.000000,0.000000
max,19.000000,8.000000,9.000000,3.000000,4.000000,2.000000,10.000000,10.000000,10.000000,1.000000


### 1.2 Target Variable Analysis

In [4]:
# Target variable distribution
print("Depression Label Distribution:")
print(df["depression_label"].value_counts())
print(f"\nClass 0 (Not Depressed): {(df['depression_label'] == 0).sum()} ({(df['depression_label'] == 0).mean()*100:.1f}%)")
print(f"Class 1 (Depressed):      {(df['depression_label'] == 1).sum()} ({(df['depression_label'] == 1).mean()*100:.1f}%)")

plt.figure(figsize=(6, 4))
ax = sns.countplot(x="depression_label", data=df, hue="depression_label", palette=["#4CAF50", "#F44336"], legend=False)
plt.xticks([0, 1], ["Not Depressed", "Depressed"])
plt.title("Distribution of Depression Label (Target)")
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
os.makedirs('screenshots', exist_ok=True)
plt.savefig('screenshots/Class_Distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Depression Label Distribution:
depression_label
0    1169
1      31
Name: count, dtype: int64

Class 0 (Not Depressed): 1169 (97.4%)
Class 1 (Depressed):      31 (2.6%)


/tmp/ipykernel_15123/3790435496.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(x="depression_label", data=df, palette=["#4CAF50", "#F44336"])


/tmp/ipykernel_15123/3790435496.py:17: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 1.3 Correlation Analysis

In [5]:
# Encode categorical features for correlation
df_corr = df.copy()
for col in ["gender", "platform_usage", "social_interaction_level"]:
    df_corr[col] = LabelEncoder().fit_transform(df_corr[col])

# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = df_corr.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig('screenshots/Correlation_Heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
target_corr = corr["depression_label"].drop("depression_label").sort_values(ascending=False)
print("\nTop features correlated with depression_label:")
print(target_corr)


Top features correlated with depression_label:
daily_social_media_hours    0.175201
stress_level                0.170474
anxiety_level               0.169566
platform_usage              0.018264
age                         0.010973
social_interaction_level    0.005110
academic_performance        0.001441
addiction_level            -0.013952
screen_time_before_sleep   -0.016502
physical_activity          -0.017598
gender                     -0.019836
sleep_hours                -0.190630
Name: depression_label, dtype: float64


/tmp/ipykernel_15123/3948625478.py:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 1.4 Feature Distributions

In [6]:
num_features = ["age", "daily_social_media_hours", "sleep_hours", "screen_time_before_sleep",
                "academic_performance", "physical_activity", "stress_level", "anxiety_level",
                "addiction_level"]

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.flatten()
for i, feat in enumerate(num_features):
    sns.histplot(df[feat], bins=20, kde=True, ax=axes[i], color="teal")
    axes[i].set_title(feat.replace("_", " ").title())
plt.tight_layout()
plt.savefig('screenshots/Feature_Distributions.png', dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_15123/173846524.py:11: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 2. Data Preprocessing

### 2.1 Check for Missing Values

In [7]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

Missing values per column:
age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64

Total missing values: 0


### 2.2 Handle Categorical Variables
We use Label Encoding because:
- `gender` is binary (male/female) — ordinal encoding works fine
- `platform_usage` has 3 categories (Instagram/TikTok/Both) with a natural ordering
- `social_interaction_level` has 3 categories (low/medium/high) with ordinal meaning
Label Encoding keeps dimensionality low, which is helpful for distance-based KNN.

In [8]:
# Handle categorical variables - encode them numerically
df_encoded = df.copy()

# Label encode categorical columns
label_encoders = {}
categorical_columns = ['gender', 'platform_usage', 'social_interaction_level']

for col in categorical_columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

print("Categorical encoding completed!")
for col, le in label_encoders.items():
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

Categorical encoding completed!
gender: {'female': 0, 'male': 1}
platform_usage: {'Both': 0, 'Instagram': 1, 'TikTok': 2}
social_interaction_level: {'high': 0, 'low': 1, 'medium': 2}


### 2.3 Train-Validation-Test Split
We use a **stratified split** to preserve the class distribution in all subsets, which is critical given the severe class imbalance (only ~2.5% positive samples).

In [9]:
# Prepare features and target
X = df_encoded.drop('depression_label', axis=1)
y = df_encoded['depression_label']

# Stratified split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
# Split temp: 50% validation, 50% test (15% each of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training samples:   {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation samples: {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test samples:       {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining set class distribution:")
print(y_train.value_counts().to_dict())

Training samples:   840 (70.0%)
Validation samples: 180 (15.0%)
Test samples:       180 (15.0%)

Training set class distribution:
{0: 818, 1: 22}


### 2.4 Feature Scaling
KNN is a distance-based algorithm — features with larger scales would dominate the distance calculation. StandardScaler (Z-score normalization) ensures all features contribute equally.

In [10]:
# Standardize features (mandatory for distance-based KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"X_train_scaled mean: {X_train_scaled.mean():.6f}")
print(f"X_train_scaled std:  {X_train_scaled.std():.6f}")

Feature scaling completed!
X_train_scaled mean: 0.000000
X_train_scaled std:  1.000000


## 3. KNN Model Training with Hyperparameter Tuning
We use **GridSearchCV** with StratifiedKFold cross-validation to find the optimal:
- `n_neighbors` (k): number of neighbors
- `weights`: uniform vs distance-weighted voting
- `metric`: distance metric (euclidean, manhattan, minkowski)

Due to the extreme class imbalance, we use `F1-score` as the scoring metric rather than accuracy.

In [11]:
# Define parameter grid
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15, 17, 19],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

# Initialize KNN
knn_base = KNeighborsClassifier()

# Grid search with stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=knn_base,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# Train
print("Running GridSearchCV for KNN hyperparameter tuning...")
grid_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation F1-score: {grid_search.best_score_:.4f}")

# Best model
best_knn = grid_search.best_estimator_

Running GridSearchCV for KNN hyperparameter tuning...
Fitting 5 folds for each of 54 candidates, totalling 270 fits



Best parameters: {'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'uniform'}
Best cross-validation F1-score: 0.1810


### 3.1 Elbow Method Plot (Error Rate vs. K-Value)

In [12]:
# Elbow Method: evaluate error rate for different k values
# Using best weights and metric from GridSearch
best_params = grid_search.best_params_
k_range = range(1, 21)
error_rates = []

for k in k_range:
    knn_temp = KNeighborsClassifier(
        n_neighbors=k,
        weights=best_params['weights'],
        metric=best_params['metric']
    )
    knn_temp.fit(X_train_scaled, y_train)
    y_pred_temp = knn_temp.predict(X_val_scaled)
    error_rates.append(1 - accuracy_score(y_val, y_pred_temp))

# Plot Elbow Method
plt.figure(figsize=(10, 6))
plt.plot(k_range, error_rates, 'bo-', markersize=6, linewidth=2)
plt.axvline(x=best_params['n_neighbors'], color='red', linestyle='--',
            label=f"Optimal k = {best_params['n_neighbors']} (from GridSearch)")
plt.xlabel('Number of Neighbors (k)', fontsize=12)
plt.ylabel('Error Rate', fontsize=12)
plt.title('Elbow Method: Error Rate vs. K-Value', fontsize=14)
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('screenshots/Elbow_Method.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Optimal k from GridSearchCV: {best_params['n_neighbors']}")
print(f"Optimal weights: {best_params['weights']}")
print(f"Optimal metric: {best_params['metric']}")

Optimal k from GridSearchCV: 3
Optimal weights: uniform
Optimal metric: manhattan


/tmp/ipykernel_15123/1619788086.py:30: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 4. Model Evaluation

In [13]:
# Predict on test set
y_pred = best_knn.predict(X_test_scaled)
y_prob = best_knn.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("=" * 55)
print("KNN MODEL EVALUATION")
print("=" * 55)
print(f"Accuracy:    {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision:   {prec:.4f}")
print(f"Recall:      {rec:.4f}")
print(f"F1-Score:    {f1:.4f}")
print(f"AUC-ROC:     {auc:.4f}")
print(f"Best K:      {best_params['n_neighbors']}")
print(f"Weights:     {best_params['weights']}")
print(f"Metric:      {best_params['metric']}")
print("=" * 55)

KNN MODEL EVALUATION
Accuracy:    0.9778 (97.78%)
Precision:   0.0000
Recall:      0.0000
F1-Score:    0.0000
AUC-ROC:     0.6080
Best K:      3
Weights:     uniform
Metric:      manhattan


### 4.1 Confusion Matrix

In [14]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Depressed (0)', 'Depressed (1)'],
            yticklabels=['Not Depressed (0)', 'Depressed (1)'],
            annot_kws={'size': 14})
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix - KNN', fontsize=14)
plt.tight_layout()
plt.savefig('screenshots/Confusion_Matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (Not Depressed correct):  {tn}")
print(f"False Positives (Depressed wrong):       {fp}")
print(f"False Negatives (Not Depressed wrong):   {fn}")
print(f"True Positives (Depressed correct):      {tp}")

True Negatives (Not Depressed correct):  176
False Positives (Depressed wrong):       0
False Negatives (Not Depressed wrong):   4
True Positives (Depressed correct):      0


/tmp/ipykernel_15123/2216721693.py:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 4.2 ROC Curve

In [15]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'KNN (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.2)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - KNN', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('screenshots/ROC_Curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC = {auc:.4f}")
print("AUC of 1.0 = Perfect classifier")
print("AUC of 0.5 = Random classifier")

AUC = 0.6080
AUC of 1.0 = Perfect classifier
AUC of 0.5 = Random classifier


/tmp/ipykernel_15123/3560801926.py:14: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 4.3 2D Decision Boundary Plot

In [16]:
# For 2D visualization, select the two most important features
# Based on correlation analysis: anxiety_level and stress_level are top correlated
feature_names = list(X.columns)
print(f"Feature order: {feature_names}")

# Find indices of top 2 features by correlation with target
top_features = ['anxiety_level', 'stress_level']
idx1 = feature_names.index(top_features[0])
idx2 = feature_names.index(top_features[1])

# Train KNN on just these 2 features for visualization
knn_2d = KNeighborsClassifier(
    n_neighbors=best_params['n_neighbors'],
    weights=best_params['weights'],
    metric=best_params['metric']
)
knn_2d.fit(X_train_scaled[:, [idx1, idx2]], y_train)

# Plot decision boundary
plt.figure(figsize=(10, 8))
DecisionBoundaryDisplay.from_estimator(
    knn_2d, X_train_scaled[:, [idx1, idx2]],
    response_method='predict',
    cmap='coolwarm',
    alpha=0.6,
    grid_resolution=200
)
scatter = plt.scatter(
    X_test_scaled[:, idx1], X_test_scaled[:, idx2],
    c=y_test, cmap='coolwarm', edgecolors='k', s=50
)
plt.xlabel(f'{top_features[0]} (standardized)', fontsize=12)
plt.ylabel(f'{top_features[1]} (standardized)', fontsize=12)
plt.title(f'KNN Decision Boundary (k={best_params["n_neighbors"]}, {best_params["metric"]})', fontsize=14)
plt.tight_layout()
plt.savefig('screenshots/Decision_Boundary.png', dpi=150, bbox_inches='tight')
plt.show()

Feature order: ['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level']


/tmp/ipykernel_15123/1835355791.py:37: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 4.4 Sample Predictions with Probabilities

In [17]:
y_prob_full = best_knn.predict_proba(X_test_scaled)[:, 1]

sample_df = pd.DataFrame({
    'Actual': y_test[:15].values,
    'Predicted': y_pred[:15],
    'Probability (Depressed)': y_prob_full[:15]
})
sample_df['Actual Label'] = sample_df['Actual'].map({0: 'Not Depressed', 1: 'Depressed'})
sample_df['Predicted Label'] = sample_df['Predicted'].map({0: 'Not Depressed', 1: 'Depressed'})

print("Sample Predictions with Probability Scores:")
display(sample_df[['Actual Label', 'Predicted Label', 'Probability (Depressed)']])

correct = (y_pred == y_test.values).sum()
print(f"\nCorrect predictions: {correct}/{len(y_test)} ({correct/len(y_test)*100:.1f}%)")

Sample Predictions with Probability Scores:


,Actual Label,Predicted Label,Probability (Depressed)
0,Not Depressed,Not Depressed,0.000000
1,Not Depressed,Not Depressed,0.000000
2,Not Depressed,Not Depressed,0.000000
3,Not Depressed,Not Depressed,0.000000
4,Not Depressed,Not Depressed,0.000000
5,Not Depressed,Not Depressed,0.000000
6,Not Depressed,Not Depressed,0.000000
7,Not Depressed,Not Depressed,0.000000
8,Not Depressed,Not Depressed,0.000000
9,Not Depressed,Not Depressed,0.000000



Correct predictions: 176/180 (97.8%)


## 5. Summary

### KNN Model Results
- **Best K Value**: 3 (manhattan distance, uniform weights)
- **Test Accuracy**: 97.78% (misleading due to imbalance)
- **Test F1-Score**: 0.0000
- **AUC-ROC**: 0.6080

### Key Insights
1. KNN is sensitive to feature scaling — StandardScaler was essential for fair distance computation
2. The extreme class imbalance (~2.5% positive) makes accuracy a misleading metric; F1-score is more informative
3. KNN with k=3 predicts all samples as "Not Depressed" — the minority class is too sparse for nearest-neighbor voting
4. GridSearchCV with F1-scoring correctly identified optimal hyperparameters, but the fundamental limitation of KNN on imbalanced data remains
5. SVM achieves significantly better results (AUC=0.96 vs 0.61) on this dataset due to its margin-based optimization with class weighting

In [18]:
# Print classification report for detailed per-class metrics
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Depressed', 'Depressed'], zero_division=0))

Classification Report:
               precision    recall  f1-score   support

Not Depressed       0.98      1.00      0.99       176
    Depressed       0.00      0.00      0.00         4

     accuracy                           0.98       180
    macro avg       0.49      0.50      0.49       180
 weighted avg       0.96      0.98      0.97       180



/home/ahsan/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ahsan/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ahsan/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
